In [ ]:
!pip install pandas pytube moviepy
!pip install pydub
!pip install pytube

In [ ]:
# Optional Colab-only setup: mount your own Google Drive if needed.
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os
PROJECT_DIR = Path(os.environ.get('SJK_PORTFOLIO_DIR', '/content/sjk_portfolio')) / 'Yeti_vid_scrape'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
path = str(PROJECT_DIR)


In [ ]:
import re
import os
import csv
import pandas as pd
from pytube import Playlist, YouTube

YOUTUBE_STREAM_AUDIO = '140'
url = str(PROJECT_DIR / 'Yeti_Sports_Med.csv')
DOWNLOAD_DIR = str(PROJECT_DIR / 'mp4')
file_path = os.path.join(DOWNLOAD_DIR, 'metadata.csv')
if not os.path.exists(file_path):
    with open(file_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Title', 'Video ID', 'Channel ID', 'Description', 'Length (sec)', 'Date Uploaded'])

data = []
existing_data = pd.read_csv(file_path)

def sanitize_filename(title):
    return "".join(c for c in title if c.isalnum() or c in ' -_.')

# assuming the file is named 'urls.csv'
with open(url, 'r') as csvfile:
    url_reader = csv.reader(csvfile)
    count = 1  # start with 1
    for url in url_reader:
        yt = YouTube(url[0])
        sanitized_title = sanitize_filename(yt.title)
        filename = f"{count:03d}-{sanitized_title}"  # format the count to be three digits and add it to the title
        if yt.video_id in existing_data['Video ID'].values:
            print(f"{yt.title} already exists in the file. Checking for audio.")
            audio_file_path = os.path.join(DOWNLOAD_DIR, f"{filename}.mp4")
            if os.path.exists(audio_file_path):
                print(f"Audio for video: {yt.title} already exists.")
            else:
                print(f"Audio for video: {yt.title} not found. Downloading")
                yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{filename}.mp4")
        else:
            data.append([yt.title, yt.video_id, yt.channel_id, yt.description, yt.length, yt.publish_date])
            yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{filename}.mp4")
            print(yt.video_id)
        count += 1  # increment the count for the next video

df = pd.DataFrame(data,columns=['Title', 'Video ID', 'Channel ID', 'Description', 'Length (sec)', 'Date Uploaded'])
df.to_csv(file_path, mode='a', index=False, header=False)

print("The videos have been downloaded!")


In [ ]:
import subprocess

DOWNLOAD_DIR = str(PROJECT_DIR / 'mp4')
folder = str(PROJECT_DIR)
mp3_folder = os.path.join(folder, "mp3")

if not os.path.exists(mp3_folder):
    os.makedirs(mp3_folder)

for filename in os.listdir(DOWNLOAD_DIR):
    if filename.endswith(".mp4"):
        mp4_file = os.path.join(DOWNLOAD_DIR, filename)
        mp3_file = os.path.join(mp3_folder, filename[:-4] + ".mp3")
        subprocess.call(['ffmpeg', '-i', mp4_file, mp3_file])